# **Build Interactive LLM Agents with Tools**

## Setup

*   [`langchain-openai`](https://pypi.org/project/langchain-openai/) is a partner package of LangChain and integrates OpenAI LLMs to the framework.

### Installing Required Libraries

In [ ]:
%pip install langchain===0.3.25 | tail -n 1
%pip install langchain-openai===0.3.19 | tail -n 1

### Importing Required Libraries

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

## Creating Custom Tools with LangChain

### Anatomy of a tool

Let's provide the basic building blocks of a tool, consider the following tool:

```python
@tool
def tool_name(input_param: input_type) -> output_type:
   """
   Clear description of what the tool does.
   
   Args:
       input_param (input_type): Description of this parameter
   
   Returns:
       output_type: Description of what is returned
   """
   # Function implementation
   result = process(input_param)
   return result
```

### Key components

**@tool decorator**
   - Registers the function with LangChain
   - Creates tool attributes (.name, .description, .func)
   - Generates JSON schema for validation
   - Transforms regular functions into callable tools

**Function name**
   - Used by LLM to select appropriate tool
   - Used as reference in chains and tool mappings
   - Appears in tool call logs for debugging
   - Should clearly indicate the tool's purpose

**Type annotations**
   - Enable automatic input validation
   - Create schema for parameters
   - Allow proper serialization of inputs/outputs
   - Help LLM understand required input formats

**Docstring**
   - Provides context for the LLM to decide when to use the tool
   - Documents parameter requirements
   - Explains expected outputs and behavior
   - Is critical for tool selection by the LLM

6. **Implementation**
   - Executes the actual operation
   - Handles errors appropriately
   - Returns properly formatted results
   - Should be efficient and robust

### Defining an add function


In [ ]:
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b

The decorator wraps the `add()` function in LangChain's predefined tool schema. See more about defining custom LangChain tools [here](https://python.langchain.com/docs/how_to/custom_tools/).

### Add tools to the LLM

Let's connect and bind the function to the chat model.

In [ ]:
tools = [add]

llm_with_tools = llm.bind_tools(tools)

Use the `bind_tools(tools)` method to connect a list of tools to the LLM for use. From now on, whenever the call is invoked, the model (with tools) will recognize and use the add tool whenever it needs to compute a sum.

### Create more tools

Let's create some more basic arithmetic tools.

In [ ]:
@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b

### Testing the functions

Let's setup a way to test your tools.

In [ ]:
tool_map = {
    "add": add, 
    "subtract": subtract,
    "multiply": multiply
}

input_ = {
    "a": 1,
    "b": 2
}

tool_map["add"].invoke(input_)

### Add new tools to LLM

Let's add all three tools to the LLM.

In [ ]:
tools = [add, subtract, multiply]

llm_with_tools = llm.bind_tools(tools)

## Interacting with the Model

### Craft the user query

In [ ]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

### Invoke the model

Now let's run the model with the context (chat history) that contains the user query.

In [ ]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print(type(response_1))
#print(response_1)

### Parse tool calls

In [ ]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name:\n{tool_1_name}')
print(f'tool args:\n{tool_1_args}')
print(f'tool call ID:\n{tool_call_1_id}')

- Extracting the `name` from the first call gives the name of the tool to use.
    - `add` in this case
- Extracting the `args` gives the inputs to pass into the tool.
    - `{a: 3, b: 2}` in this case
- Extracting the `id` gives the unique identifier for the tool call
    - The ID will be different each time, linking tool calls to their respective responses
    - Crucial in differentiating calls to the same tool and parallel tool calls

### Invoke the Tool

Given the tool call details from the LLM, invoke the correct tool with the correct arguments.

In [ ]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

Use the `tool_map`, passing in the tool name and parameters to get a response. Then wrap that response in a `ToolMessage` object from LangChain along with the tool call ID. This action allows the model and LangChain to better process tool responses and overall conversation between user and model and tool. Feel free to uncomment the print statement to see what the `tool_message` looks like.

In [ ]:
chat_history.append(tool_message)

Next, append the `tool_message` to the `chat_history` so the model preserves context and sees prior conversation for a better conversing experience. Now the chat history contains a `HumanMessage` (initial user query), an `AIMessage` (the response from the model), and a `ToolMessage` (the output of the tool).

### Generate a final answer from chat history

As a final step, pass the entire `chat_history` into the LLM one more time to get a final response.

In [ ]:
answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

## Building an Agent

In [ ]:
class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.contet # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

This agent does the exact same process as above except the interaction with the model handling is all contained within the `run()` method.

In [ ]:
my_agent = ToolCallingAgent(llm)

print(my_agent.run("one plus 2"))

print(my_agent.run("one - 2"))

print(my_agent.run("three times two"))

Here are three examples of the agent in use. These agents are dynamic and can handle many different types and formats of data as input. This capability is a major benefit of AI agents as the input data doesn't need to be normalized or formatted a certain way.